# 10 - Capstone Robot Stack

## What / Why / How

**What we are trying to do:** Integrate planning, control, learning, evaluation, and safety thinking into one small robot stack.

**Why this matters:** Robotics mastery comes from combining pieces. A good robot is a system, not just a controller or a model.

**How we will do it:** Plan a grid route, follow it under biased dynamics, learn a residual correction from data, and compare metrics across runs.

## Goal

Build a small integrated robot stack:

- Plan a route with A*.
- Follow waypoints with feedback control.
- Add noisy motion.
- Learn a residual correction from demonstrations.
- Evaluate success and failure.

This is the shape of real robotics work: not one clever model, but a system with planning, control, estimation, learning, and safety checks.

In [ ]:
import math
import random
from collections import defaultdict

import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see the plot: pip install -r requirements.txt")

## Planning Layer

In [ ]:
import heapq

H, W = 22, 22
grid = np.zeros((H, W), dtype=int)
grid[6:17, 8] = 1
grid[4, 8:18] = 1
grid[14, 3:14] = 1
grid[10:20, 17] = 1
start = (2, 2)
goal = (19, 20)

def astar_grid(grid, start, goal):
    H, W = grid.shape
    def h(a, b):
        return abs(a[0]-b[0]) + abs(a[1]-b[1])
    def nbrs(s):
        r, c = s
        for dr, dc in [(1,0), (-1,0), (0,1), (0,-1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < H and 0 <= nc < W and grid[nr, nc] == 0:
                yield (nr, nc)
    frontier = [(0, start)]
    came = {start: None}
    cost = {start: 0}
    while frontier:
        _, cur = heapq.heappop(frontier)
        if cur == goal:
            break
        for nxt in nbrs(cur):
            new = cost[cur] + 1
            if nxt not in cost or new < cost[nxt]:
                cost[nxt] = new
                heapq.heappush(frontier, (new + h(nxt, goal), nxt))
                came[nxt] = cur
    path = []
    cur = goal
    while cur in came and cur is not None:
        path.append(cur)
        cur = came[cur]
    return path[::-1]

path = astar_grid(grid, start, goal)
waypoints = np.array(path, dtype=float)
print("planned cells:", len(path))

## Controller And Learned Residual

In [ ]:
rng = np.random.default_rng(11)
hidden_bias = np.array([0.06, -0.04])  # unmodeled drift in row/col coordinates

def limit_norm(vec, max_norm=0.45):
    n = np.linalg.norm(vec)
    if n > max_norm:
        return vec * (max_norm / n)
    return vec

def residual_features(error):
    return np.array([error[0], error[1], 1.0])

# Demonstrations tell us the correction needed to cancel the hidden bias.
X_demo = []
Y_demo = []
for _ in range(200):
    error = rng.normal(0, 2.0, size=2)
    X_demo.append(residual_features(error))
    Y_demo.append(-hidden_bias + rng.normal(0, 0.01, size=2))
X_demo = np.array(X_demo)
Y_demo = np.array(Y_demo)
W_residual, *_ = np.linalg.lstsq(X_demo, Y_demo, rcond=None)

def learned_residual(error):
    return residual_features(error) @ W_residual

print("learned residual for zero error:", learned_residual(np.array([0.0, 0.0])))
print("true correction:", -hidden_bias)

## Closed-Loop Simulation

In [ ]:
def is_collision(pos):
    cell = tuple(np.round(pos).astype(int))
    r, c = cell
    if r < 0 or r >= H or c < 0 or c >= W:
        return True
    return grid[r, c] == 1

def follow_path(use_residual=False, seed=0, max_steps=450):
    local_rng = np.random.default_rng(seed)
    pos = np.array(start, dtype=float)
    traj = [pos.copy()]
    waypoint_idx = 0
    collided = False

    for _ in range(max_steps):
        target = waypoints[min(waypoint_idx, len(waypoints)-1)]
        error = target - pos
        if np.linalg.norm(error) < 0.35 and waypoint_idx < len(waypoints) - 1:
            waypoint_idx += 1
            target = waypoints[waypoint_idx]
            error = target - pos
        u = limit_norm(0.8 * error)
        if use_residual:
            u = u + learned_residual(error)
        noise = local_rng.normal(0, 0.015, size=2)
        pos = pos + u + hidden_bias + noise
        traj.append(pos.copy())
        if is_collision(pos):
            collided = True
            break
        if np.linalg.norm(pos - np.array(goal, dtype=float)) < 0.6:
            break
    return np.array(traj), collided

nominal_traj, nominal_collision = follow_path(use_residual=False, seed=1)
learned_traj, learned_collision = follow_path(use_residual=True, seed=1)

def report(name, traj, collided):
    final_error = np.linalg.norm(traj[-1] - np.array(goal, dtype=float))
    print(name, "steps", len(traj), "final_error", round(float(final_error), 3), "collided", collided)

report("nominal", nominal_traj, nominal_collision)
report("learned residual", learned_traj, learned_collision)

In [ ]:
if HAS_PLOT:
    canvas = grid.astype(float)
    for r, c in path:
        canvas[r, c] = 0.35
    plt.figure(figsize=(6, 6))
    plt.imshow(canvas, origin="lower", cmap="gray_r")
    plt.plot(nominal_traj[:, 1], nominal_traj[:, 0], label="nominal", color="tab:orange")
    plt.plot(learned_traj[:, 1], learned_traj[:, 0], label="learned residual", color="tab:blue")
    plt.scatter([start[1], goal[1]], [start[0], goal[0]], c=["tab:green", "tab:red"], s=80)
    plt.legend()
    plt.title("Capstone robot stack")
    plt.show()
else:
    plot_unavailable()

## Evaluation Over Multiple Seeds

In [ ]:
def evaluate(use_residual, n=40):
    successes = 0
    collisions = 0
    errors = []
    for seed in range(n):
        traj, collided = follow_path(use_residual=use_residual, seed=seed)
        err = np.linalg.norm(traj[-1] - np.array(goal, dtype=float))
        successes += (err < 0.8 and not collided)
        collisions += collided
        errors.append(err)
    return {
        "success_rate": successes / n,
        "collision_rate": collisions / n,
        "mean_final_error": float(np.mean(errors)),
    }

print("nominal:", evaluate(False))
print("learned residual:", evaluate(True))

## Capstone Extensions

Choose one:

1. Add a Kalman filter from notebook 4 so the controller uses an estimate instead of true position.
2. Replace A* with RRT from notebook 6.
3. Train the residual on biased trajectories instead of directly revealing the correction.
4. Add a safety layer that rejects actions leading into occupied cells.
5. Port the idea to ROS 2 with nodes for planner, controller, estimator, and logger.
6. Replace the residual with a small neural policy once you install the advanced requirements.

## Mastery Checklist

You are ready to move into real robot projects when you can explain and implement:

- Coordinate frames and transformations.
- Forward and inverse kinematics.
- Feedback control with actuator limits.
- State estimation under sensor noise.
- Occupancy grids and localization.
- Graph search and sampling-based planning.
- Behavior cloning and action chunking.
- Basic RL and why sim-to-real is hard.
- Why modern robot policies use large datasets, multimodal inputs, and distributional action models.
- How to evaluate robot behavior safely.